# 拓扑码的神经解码器——独立复现

本 Notebook 独立复现 G. Torlai 和 R. G. Melko 的论文 *Neural Decoder for Topological Codes*，发表于 **Physical Review Letters 119**，030501（2017），[doi:10.1103/PhysRevLett.119.030501](https://doi.org/10.1103/PhysRevLett.119.030501)。论文 PDF 保存在 `paper/docs/`。

**职责分工。** 本 Notebook 是实验的程序入口，负责阶段顺序，以及 split、epoch、minibatch、Gibbs 采样步和测试样本这一层的循环。更底层的操作均调用 `ai_qec/`：Toric code 与噪声模型、RBM 采样和 CD-k 更新、syndrome 兼容性检查、精确 MWPM、数据集写入与校验、指标和报告构建，以及 run 记录。

| 论文算法 1 的步骤 | Notebook 章节 |
|---|---|
| 第 1–2 行：`e₀`、`S₀ = S(e₀)` | 第 3 节：数据生成 |
| 第 3 行：`RBM = {e, S=S₀, h}`（通过 CD-k 训练） | 第 4 节：训练 |
| 第 4–8 行：钳制 syndrome，执行 Gibbs 采样直到 `S(e) = S₀`，令 `r = e` | 第 5 节：神经解码 |
| 根据 `e₀ ⊕ r` 判断逻辑失败 | 第 6 节：基准对照 |

下方单元格内的参数只用于小规模冒烟验证。其输出可以验证实验路径，但不能视为论文最终图表的数值复现，也不足以支持阈值结论。

In [4]:
# 导入共享 setup，并由已安装的 ai_qec 包定位项目源码根目录。
from __future__ import annotations

from pathlib import Path
import time

import matplotlib.pyplot as plt
import numpy as np
import torch

import ai_qec.notebook_api as qec

PROJECT_ROOT = qec.find_project_root()

## 1. 实验配置契约

本节定义 `EXPERIMENT_CONFIG`：码、噪声、数据、模型及训练参数，检查它选择了本 Notebook 支持的数据生成器和模型，构建 `code` 与 `noise`，并记录参数哈希。下一节启动 run 前会核对该哈希，防止配置在对象构建后被改动。本 Notebook 不读取 `configs/` 下的 YAML。脚本 runner 的 `flow` 在此由 Notebook 的阶段顺序取代。

需要检查并行链时，可在 `EXPERIMENT_CONFIG` 中将 `training.decoder.parallel_chains` 从 `1` 改为 `64`，再开始一次新的 run；该结果应单独标为并行链实验。

In [5]:
# 定义码、噪声、数据、模型和训练参数。
EXPERIMENT_CONFIG = {
    'qec': {'code': 'toric_code', 'distance': 4, 'rounds': 1, 'task': 'code_capacity'},
    'noise': {'model': 'phase_flip', 'p_error': 0.08},
    'data': {
        'generator': 'toric_code_capacity',
        'train_samples': 256, 'validation_samples': 64, 'test_samples': 24,
        'dataset_id': 'torlai_melko_2017_l4_p008_smoke',
        'output_dir': 'datasets', 'batch_size': 64,
        'preprocessing': {'representation': 'error_syndrome'},
    },
    'model': {'implementation': 'joint_error_syndrome_rbm', 'hidden_units': 16, 'init_width': 0.01},
    'training': {
        'trainer': 'rbm_cd', 'device': 'cuda', 'epochs': 8, 'batch_size': 64,
        'learning_rate': 0.05, 'cd_steps': 1, 'weight_decay': 0.0001,
        'decoder': {'burn_in': 25, 'max_steps': 400, 'parallel_chains': 1},
    },
}
qec.require_experiment_kind(
    EXPERIMENT_CONFIG, generator='toric_code_capacity', model='joint_error_syndrome_rbm',
)
code = qec.build_code(EXPERIMENT_CONFIG)
noise = qec.build_noise_model(EXPERIMENT_CONFIG)
EXPERIMENT_CONFIG_HASH = qec.config_hash(EXPERIMENT_CONFIG)

## 2. 校验配置并创建独立 run

本节定义 `RUN_CONFIG`：schema、执行环境、实验标识、研究方向、可复现设置和输出位置。`qec.start_notebook_run(...)` 先核对第 1 节已构建对象所用的实验参数，合并并严格校验两组配置，检查内核与设备，展示完整有效配置；全部通过后才创建唯一 run，并将有效配置保存为 `config.yaml`。后续阶段通过 `run.stage(...)` 记录状态、耗时及产物哈希。

In [6]:
# 校验配置并创建本次执行的唯一 run。
PAPER_PATH = PROJECT_ROOT / 'paper' / 'docs' / 'A Neural Decoder for Topological Codes.pdf'
assert PAPER_PATH.is_file(), PAPER_PATH

RUN_CONFIG = {
    'schema_version': 1,
    'execution': {'conda_env': 'quantum'},
    'experiment': {
        'name': 'torlai_melko_2017_rbm_smoke',
        'title': 'Joint RBM decoder for a toric-code capacity experiment',
        'description': 'Small-lattice smoke of the Torlai--Melko (2017) joint RBM workflow.',
        'owner': 'researcher',
        'tags': ['ai-qec', 'toric-code', 'rbm', 'paper-reproduction', 'smoke'],
    },
    'topic': {'direction': 'D1', 'subtopic': 'D1.2', 'name': 'generative_neural_decoding'},
    'reproducibility': {
        'seeds': [17], 'deterministic': True, 'save_environment': True,
        'save_git_commit': True, 'require_clean_worktree': False,
    },
    'outputs': {'runs_root': 'runs'},
}

run, DEVICE, SEED = qec.start_notebook_run(
    RUN_CONFIG, EXPERIMENT_CONFIG, project_root=PROJECT_ROOT,
    notebook='paper/srcs/torlai_melko_2017.ipynb',
    prepared_experiment_hash=EXPERIMENT_CONFIG_HASH,
)
config, RUN_DIR = run.config, run.run_dir
RUN_DIR

配置来源：paper/srcs/torlai_melko_2017.ipynb；以下是校验后的完整有效配置
schema_version: 1
execution:
  conda_env: quantum
experiment:
  name: torlai_melko_2017_rbm_smoke
  title: Joint RBM decoder for a toric-code capacity experiment
  description: Small-lattice smoke of the Torlai--Melko (2017) joint RBM workflow.
  owner: researcher
  tags:
  - ai-qec
  - toric-code
  - rbm
  - paper-reproduction
  - smoke
topic:
  direction: D1
  subtopic: D1.2
  name: generative_neural_decoding
reproducibility:
  seeds:
  - 17
  deterministic: true
  save_environment: true
  save_git_commit: true
  require_clean_worktree: false
outputs:
  runs_root: runs
qec:
  code: toric_code
  distance: 4
  rounds: 1
  task: code_capacity
noise:
  model: phase_flip
  p_error: 0.08
data:
  generator: toric_code_capacity
  train_samples: 256
  validation_samples: 64
  test_samples: 24
  dataset_id: torlai_melko_2017_l4_p008_smoke
  output_dir: datasets
  batch_size: 64
  preprocessing:
    representation: error_syndrome
model:
  im

PosixPath('/home/zephy/workspace/AI for QEC/runs/20260916T130355087866Z_489a3a373df4_3c84611b')

## 3. 数据生成——论文算法 1 第 1–2 行

对每个 split，独立采样相位翻转错误链 `e₀`，并计算完美测量条件下的顶点 syndrome `S₀ = S(e₀)`。数据集按内容身份寻址且不可覆盖：如果对应生成规范的数据目录已经存在，则先校验再复用；否则将各 split 写入临时目录，只有全部写入成功后才提交为正式数据集。

In [ ]:
# 逐 split、逐 batch 采样 e0 并计算 S0；新数据在 staging 目录中写完后才原子提交，最后统一校验。
DATASET_DIR = qec.data_output_dir(config, PROJECT_ROOT)
GENERATION_BATCH = int(config['data']['batch_size'])

with run.stage('generate_data') as step:
    if DATASET_DIR.exists():
        step['reused_immutable'] = True
    else:
        with qec.staged_dataset_dir(DATASET_DIR) as staging:
            files = {}
            for split, n_samples in qec.split_sample_counts(config).items():
                rng = np.random.default_rng(SEED + qec.SPLIT_SEED_OFFSETS[split])
                errors, syndromes = [], []
                for start in range(0, n_samples, GENERATION_BATCH):
                    e0 = noise.sample_errors(min(GENERATION_BATCH, n_samples - start), code.num_data_qubits, rng)  # line 1
                    errors.append(e0)
                    syndromes.append(code.syndrome(e0))                                                          # line 2
                files[f'{split}.npz'] = qec.write_toric_split(
                    staging / f'{split}.npz', split=split, dataset_id=config['data']['dataset_id'],
                    physical_error=np.concatenate(errors), syndrome=np.concatenate(syndromes),
                    lattice_size=code.distance, p_error=noise.p_error,
                )
            qec.write_json(staging / 'dataset_manifest.json', qec.build_toric_dataset_manifest(config, code, files))
    dataset_manifest = qec.validate_toric_dataset(DATASET_DIR, config)
    step['outputs'].append(DATASET_DIR / 'dataset_manifest.json')

run.refresh_dataset()
splits = {name: qec.load_toric_split(DATASET_DIR, name) for name in ('train', 'validation', 'test')}
{
    'dataset_dir': str(DATASET_DIR.relative_to(PROJECT_ROOT)),
    'reused': step.get('reused_immutable', False),
    'sample_counts': dataset_manifest['sample_counts'],
    'mean_error_weight': {name: float(split.physical_error.sum(axis=1).mean()) for name, split in splits.items()},
}

## 4. 训练——论文算法 1 第 3 行的联合 RBM

RBM 建模可见单元 `[e | S]` 的联合分布。每个 epoch 都会打乱训练集，并对每个 minibatch 执行一次 CD-k 更新。验证集一步重建 BCE 最低的 checkpoint 保存为 `best.pt`；这一选择规则是本项目的实验约定，并非来自原论文。

In [ ]:
# CD-k 训练循环：epoch 洗牌 → minibatch 更新 → 验证集重建误差 → 保存 best/last checkpoint。
TRAIN = config['training']
EPOCHS, BATCH_SIZE, CD_STEPS = int(TRAIN['epochs']), int(TRAIN['batch_size']), int(TRAIN['cd_steps'])
BEST_CHECKPOINT = RUN_DIR / 'checkpoints' / 'best.pt'
LAST_CHECKPOINT = RUN_DIR / 'checkpoints' / 'last.pt'
SUMMARY_PATH = RUN_DIR / 'training_summary.json'

with run.stage('train') as step:
    train_visible, validation_visible = splits['train'].visible, splits['validation'].visible
    model = qec.build_model(config, error_units=code.num_data_qubits, syndrome_units=code.num_syndrome_bits).to(DEVICE)
    optimizer = torch.optim.SGD(model.parameters(), lr=float(TRAIN['learning_rate']), weight_decay=float(TRAIN['weight_decay']))
    order_rng = np.random.default_rng(SEED + 41)
    torch_rng = torch.Generator(device=DEVICE).manual_seed(SEED + 41)

    history, best_validation, best_epoch = [], float('inf'), None
    training_started = time.perf_counter()
    for epoch in range(1, EPOCHS + 1):
        order = order_rng.permutation(len(train_visible))
        batch_losses = [
            model.contrastive_divergence_step(train_visible[order[start:start + BATCH_SIZE]], optimizer=optimizer, cd_steps=CD_STEPS, generator=torch_rng)
            for start in range(0, len(order), BATCH_SIZE)
        ]
        row = {
            'epoch': epoch,
            'train_reconstruction_bce': float(np.mean(batch_losses)),
            'validation_reconstruction_bce': model.reconstruction_bce(validation_visible),
        }
        history.append(row)
        print(f"epoch {epoch:3d}  训练 BCE {row['train_reconstruction_bce']:.4f}  验证 BCE {row['validation_reconstruction_bce']:.4f}")
        if row['validation_reconstruction_bce'] < best_validation:
            best_validation, best_epoch = row['validation_reconstruction_bce'], epoch
            model.save(BEST_CHECKPOINT, optimizer=optimizer, metadata=qec.rbm_checkpoint_metadata(
                config, dataset_manifest, PROJECT_ROOT, metrics=row, selection=qec.BEST_CHECKPOINT_SELECTION))

    model.save(LAST_CHECKPOINT, optimizer=optimizer, metadata=qec.rbm_checkpoint_metadata(
        config, dataset_manifest, PROJECT_ROOT, metrics=history[-1], selection='final epoch'))
    training_summary = qec.rbm_training_summary(
        history, selected_epoch=best_epoch, training_time_seconds=time.perf_counter() - training_started, dataset_dir=DATASET_DIR)
    qec.write_json(SUMMARY_PATH, training_summary)
    step['outputs'] += [BEST_CHECKPOINT, LAST_CHECKPOINT, SUMMARY_PATH]

training_summary['metrics']

In [ ]:
# 绘制训练与验证集的一步重建 BCE 曲线，并标出被选中的 epoch。
fig, ax = plt.subplots(figsize=(6, 3.5))
epochs = [row['epoch'] for row in history]
ax.plot(epochs, [row['train_reconstruction_bce'] for row in history], marker='o', label='训练')
ax.plot(epochs, [row['validation_reconstruction_bce'] for row in history], marker='o', label='验证')
ax.axvline(best_epoch, color='grey', linestyle='--', linewidth=1, label=f'选中 epoch {best_epoch}')
ax.set_xlabel('epoch')
ax.set_ylabel('一步重建 BCE')
ax.legend()
ax.grid(alpha=.25)
plt.show()

## 5. 神经解码——论文算法 1 第 3–8 行

将 syndrome 单元钳制为测得的 `S₀`，交替执行块 Gibbs 更新 `h ~ p(h | e, S₀)` 与 `e ~ p(e | h)`，直到采样得到满足 `S(e) = S₀` 的错误链，并以该链作为恢复链 `r`。论文正文也说明了固定采样步数截止和失败计数；这里用 `burn_in` 与 `max_steps` 明确配置预算：仅在预热后检查兼容性，超出预算则记为超时和逻辑失败。当前算法选取第一条兼容链，并不比较各同调类的概率；`parallel_chains > 1` 是另行标注的扩展实验。

In [ ]:
# Algorithm 1 第 3–8 行：syndrome 钳制的块 Gibbs 采样循环。
DECODER = TRAIN['decoder']
BURN_IN, MAX_STEPS, PARALLEL_CHAINS = int(DECODER['burn_in']), int(DECODER['max_steps']), int(DECODER.get('parallel_chains', 1))
if not 0 <= BURN_IN < MAX_STEPS or PARALLEL_CHAINS < 1:
    raise ValueError('参数必须满足 0 <= burn_in < max_steps 且 parallel_chains >= 1')
PARITY_CHECK = torch.as_tensor(code.parity_check_matrix(), dtype=torch.int64, device=DEVICE)


@torch.no_grad()
def neural_decode(rbm, s0: np.ndarray, generator: torch.Generator) -> tuple[np.ndarray | None, int]:
    target = torch.as_tensor(s0, dtype=torch.int64, device=DEVICE)            # line 3: clamp S = S0
    e = rbm.random_error_chains(PARALLEL_CHAINS, generator)
    for sweep in range(1, MAX_STEPS + 1):                                     # line 4: while S(e) != S0 (bounded)
        h = rbm.sample_hidden(e, target, generator)                           # line 5: h ~ p(h | e, S0)
        e = rbm.sample_error(h, generator)                                    # line 6: e ~ p(e | h)
        if sweep <= BURN_IN:
            continue
        chain = qec.first_compatible_chain(e, target, PARITY_CHECK)
        if chain is not None:
            return e[chain].to(torch.uint8).cpu().numpy(), sweep              # line 8: r = e
    return None, MAX_STEPS

In [ ]:
# 在 test split 上逐样本解码；超时记为逻辑失败，结果写入 predictions 与 metrics。
PREDICTIONS_PATH = RUN_DIR / 'predictions' / 'toric_rbm_eval.npz'
METRICS_PATH = RUN_DIR / 'metrics.json'

with run.stage('evaluate') as step:
    rbm, _ = qec.load_model(config, str(BEST_CHECKPOINT))
    rbm = rbm.to(DEVICE).eval()
    test = splits['test']
    n_test = len(test.physical_error)
    recoveries = np.zeros_like(test.physical_error, dtype=np.uint8)
    recovery_valid = np.zeros(n_test, dtype=np.uint8)
    timed_out = np.zeros(n_test, dtype=np.uint8)
    gibbs_steps = np.zeros(n_test, dtype=np.int32)
    decoder_latency_ms = np.zeros(n_test, dtype=np.float64)
    logical_failure = np.ones(n_test, dtype=np.uint8)

    for index, (e0, s0) in enumerate(zip(test.physical_error, test.syndrome, strict=True)):
        generator = qec.torch_generator_from(qec.decoding_rng(config, index), DEVICE)
        started = time.perf_counter_ns()
        recovery, gibbs_steps[index] = neural_decode(rbm, s0, generator)
        decoder_latency_ms[index] = (time.perf_counter_ns() - started) / 1e6
        if recovery is None:
            timed_out[index] = 1
            continue
        recoveries[index], recovery_valid[index] = recovery, 1
        logical_failure[index] = code.logical_failure(e0[None, :], recovery[None, :])[0]

    metrics = qec.rbm_decoding_metrics(
        'test', logical_failure=logical_failure, timed_out=timed_out, recovery_valid=recovery_valid,
        gibbs_steps=gibbs_steps, decoder_latency_ms=decoder_latency_ms)
    qec.save_toric_predictions(
        PREDICTIONS_PATH, dataset=test, parallel_chains=PARALLEL_CHAINS, device=DEVICE, recovery=recoveries,
        recovery_valid=recovery_valid, timed_out=timed_out, gibbs_steps=gibbs_steps,
        decoder_latency_ms=decoder_latency_ms, logical_failure=logical_failure)
    qec.write_json(METRICS_PATH, {'metrics': metrics})
    step['outputs'] += [PREDICTIONS_PATH, METRICS_PATH]

metrics

## 6. 与精确 MWPM 的基准对照

在同一个 test split 上使用无外部依赖的精确 MWPM 参照解码器。通过 `error XOR recovery` 的同调判断逻辑失败，并用 Wilson 95% 置信区间比较失败率。精确匹配器会拒绝超过缺陷数上限的 syndrome，而不会悄悄降低求解精度；更大的格点需要可扩展的匹配后端。

In [ ]:
# MWPM 基线逐样本解码，与 RBM 结果汇总为 benchmark 报告，并结束本次 run。
with run.stage('benchmark') as step:
    mwpm = qec.ExactToricMWPMDecoder(code)
    mwpm_recoveries = np.stack([mwpm.decode(s0) for s0 in test.syndrome])
    report, benchmark_metrics = qec.build_toric_benchmark_report(
        code, split='test', p_error=noise.p_error, errors=test.physical_error, rbm_recoveries=recoveries,
        rbm_valid=recovery_valid, rbm_failures=logical_failure, decoder_latency_ms=decoder_latency_ms,
        parallel_chains=PARALLEL_CHAINS, device=DEVICE, mwpm_recoveries=mwpm_recoveries, max_exact_defects=mwpm.max_exact_defects)
    step['outputs'] += qec.write_benchmark_outputs(RUN_DIR, report, benchmark_metrics)

run.finish()
print(f'run 已完成：{RUN_DIR}')
report

In [ ]:
# 绘制 RBM Gibbs 解码器与精确 MWPM 的逻辑失败率，误差条为 Wilson 95% 置信区间。
labels = ['RBM Gibbs', '精确 MWPM']
rates = [report['rbm']['logical_error_rate'], report['mwpm_exact']['logical_error_rate']]
intervals = [report['rbm']['wilson_95'], report['mwpm_exact']['wilson_95']]
yerr = np.array([[rate - interval[0] for rate, interval in zip(rates, intervals)],
                 [interval[1] - rate for rate, interval in zip(rates, intervals)]])
fig, ax = plt.subplots(figsize=(6, 4))
ax.bar(labels, rates, yerr=yerr, capsize=5, color=['#4C78A8', '#F58518'])
ax.set_ylim(0, 1.05)
ax.set_ylabel('逻辑失败率')
ax.set_title(f"L={report['lattice_size']}, p={report['p_error']}: 冒烟 run 对照")
ax.grid(axis='y', alpha=.25)
plt.show()

In [ ]:
# 并排展示 RBM 与精确 MWPM 的同调扇区分布，便于观察逻辑失败类型。
sectors = ['00', '01', '10', '11']
x = np.arange(len(sectors))
fig, ax = plt.subplots(figsize=(6, 4))
ax.bar(x - .18, [report['rbm']['homology_counts'][s] for s in sectors], .36, label='RBM 接受的恢复链', color='#4C78A8')
ax.bar(x + .18, [report['mwpm_exact']['homology_counts'][s] for s in sectors], .36, label='精确 MWPM', color='#F58518')
ax.set_xticks(x, sectors)
ax.set_xlabel('物理错误 XOR 恢复链的同调扇区')
ax.set_ylabel('测试样本数')
ax.legend()
ax.set_title('闭合环的同调扇区')
plt.show()

## 本次 run 能说明什么

一次 Notebook 执行会生成或复用 code-capacity 数据，以 CD-k 训练联合 RBM，执行 syndrome 钳制的 Gibbs 解码，通过 `error XOR recovery` 的同调判断逻辑失败，并在同一 test split 上与精确 MWPM 对照。各阶段均记录在 `run_manifest.json` 中。冒烟配置刻意保持小规模，不能支持科学性能结论。论文规模的结论还需要预先确定格点大小、错误率和随机种子网格，采集足够多的样本，并为缺陷数较大的情况使用可扩展的匹配后端。